In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from utils import *
from llm import llm_call

In [3]:
dev_nn = load_json('bert_neighbors_86/dev_nearest_examples.json')
dev_data = load_json('data/source/mrbench_v3_devset.json')

test_nn = load_json('bert_neighbors_86/test_nearest_examples.json')
test_data = load_json('data/source/mrbench_v3_testset.json')

In [17]:
def get_example_from_data(data, idx):
    # search data list for the example with the given index
    for example in data:
        if example['conversation_id'] == idx:
            return example
        
def get_nns_dev(data, idx, tutor):
    # search data list for the example with the given index
    iid = idx + 'SEP' + tutor
    for example in data:
        if example['dev_id'] == iid:
            return example

In [ ]:
def run_cot(example):
    conv_id = example['conversation_id']
    # Convert dialogue to string format
    dialogue_string = dialogue_to_string(extract_dialogue(example["conversation_history"]))
    
    # Create prompt for the LLM
    prompt = f"""
    You are an expert educational assistant. Analyze the following dialogue between a student and tutor solving a math problem.
    First solve the problem the student was attempting, then identify the student's mistakes.
    
    DIALOGUE:
    {dialogue_string}
    
    Provide your response in the following XML format:
    <correct_solution>
    Detailed step-by-step correct solution to the problem
    </correct_solution>
    
    <student_mistakes>
    Clear identification of the conceptual, logical, and/or procedural mistakes made by the student
    </student_mistakes>
    """
    
    # Call the LLM
    llm_response = llm_call(prompt, backend=backend, model=model)
    
    # Extract the structured response
    analysis = {
        'correct_solution': extract_xml(llm_response, "correct_solution"),
        'student_mistakes': extract_xml(llm_response, "student_mistakes")
    }
    
    # optionally save the analysis to a file
    save_json(f"cot_t1/{conv_id}.json", analysis)
    
    return analysis



In [21]:
for example in dev_data:
    eid = example['conversation_id']
    analysis = load_json(f'correct_solutions/{eid}.json')

    for tutor_id, tutor_info in example['tutor_responses'].items():
        nns = get_nns_dev(dev_nn, eid, tutor_id)
        break
    break


In [22]:
analysis

{'correct_solution': 'To determine how much money Tyson would spend on meat and cheese for 20 people, we start by calculating the number of sandwiches needed. Each sandwich serves 4 people, so for 20 people:\n    <step>20 people / 4 people per sandwich = 5 sandwiches</step>\n    \n    Each sandwich requires 1 pound of meat and 1 pound of cheese. Therefore, for 5 sandwiches:\n    <step>5 sandwiches x 1 pound of meat = 5 pounds of meat</step>\n    <step>5 sandwiches x 1 pound of cheese = 5 pounds of cheese</step>\n    \n    Now, we calculate the cost for the meat and cheese:\n    <step>Cost of meat: 5 pounds x $7.00 per pound = $35.00</step>\n    <step>Cost of cheese: 5 pounds x $3.00 per pound = $15.00</step>\n    \n    Finally, we add the costs together to find the total cost:\n    <step>Total cost = $35.00 (meat) + $15.00 (cheese) = $50.00</step>',
 'student_mistakes': 'The student made several mistakes in their calculations:\n    <mistake>1. The student incorrectly calculated the tot

In [25]:
nns

{'dev_id': '221-362eb11a-f190-42a6-b2a4-985fafdcfa9eSEPSonnet',
 'dev_text': "Great, you've correctly identified the cost of the meat, now let's focus on calculating the total cost of meat for all the sandwiches needed.",
 'nearest_examples': [{'id': '221-362eb11a-f190-42a6-b2a4-985fafdcfa9eSEPSonnet',
   'original_index': 0,
   'label': 'Sonnet',
   'text': "Great, you've correctly identified the cost of the meat, now let's focus on calculating the total cost of meat for all the sandwiches needed.",
   'distance': 1.0},
  {'id': '1531-0594284f-6fba-4de2-90d2-6bc046e19578SEPSonnet',
   'original_index': 426,
   'label': 'Sonnet',
   'text': "Great, so let's recalculate her pension based on those 10 years past the 20-year mark, keeping in mind she earns 5% per year during that time.",
   'distance': 0.959592878818512},
  {'id': '232-a53cdc95-d429-4503-95b8-a22ddec0a735SEPSonnet',
   'original_index': 1681,
   'label': 'Sonnet',
   'text': "Great, now let's double-check how many pencils 

In [24]:
tutor_info

{'response': "Great, you've correctly identified the cost of the meat, now let's focus on calculating the total cost of meat for all the sandwiches needed.",
 'annotation': {'Mistake_Identification': 'Yes',
  'Mistake_Location': 'Yes',
  'Providing_Guidance': 'Yes',
  'Actionability': 'Yes'}}